In [1]:
import os
import sys
import json
from collections import Counter
from datetime import datetime, timezone, timedelta
from itertools import combinations
from pathlib import Path

import numpy as np
import polars as pl
from tqdm.notebook import tqdm
sys.path.append(os.path.abspath("../.."))

from src.utils.target_encoding import target_encoding

### Configuration

In [26]:
ID = "044"
SEED = 42
LEVEL = "l1"
FEATURE_DIR = Path(f"../../artifacts/features/{ID}")

os.makedirs(FEATURE_DIR, exist_ok=True)

pl.Config.set_tbl_rows(500)
pl.Config.set_tbl_cols(500)

polars.config.Config

### Urils

In [3]:
def check_info(
    train: pl.DataFrame,
    test: pl.DataFrame
) -> tuple[float, float, float]:
    train_mem = sum(train[col].to_numpy().nbytes for col in train.columns) / 1024**3
    test_mem = sum(test[col].to_numpy().nbytes for col in test.columns) / 1024**3

    print("=== Shape & Memory ===")
    print(f"Train Shape: {train.shape}, Test Shape: {test.shape}")
    print(f"Train Memory: {train_mem:.2f} GB, Test Memory: {test_mem:.2f} GB\n")

    dtype_counts = Counter([str(dt) for dt in train.dtypes])

    n_cat = None
    print("=== DTypes ===")
    for dtype, cnt in dtype_counts.items():
        print(f"{dtype}: {cnt}")
        if dtype == "Categorical":
            n_cat = cnt
    return train_mem, test_mem, n_cat


def downcast(df: pl.DataFrame) -> pl.DataFrame:
    INT32_MIN, INT32_MAX = -2_147_483_648, 2_147_483_647

    df = df.with_columns(pl.col(pl.Float64).cast(pl.Float32))

    # Int64で安全に落とせる列だけ選別
    int64_cols = [c for c, dt in df.schema.items() if dt == pl.Int64]
    safe_cols = []
    for c in int64_cols:
        mn, mx = df[c].min(), df[c].max()
        if mn >= INT32_MIN and mx <= INT32_MAX:
            safe_cols.append(c)

    # 安全な列だけ Int32 に
    if safe_cols:
        df = df.with_columns(pl.col(safe_cols).cast(pl.Int32))
    return df

### Feature Engineering

In [4]:
# === Load Data ===
train = pl.read_csv("../../input/train.csv").drop("id")
test = pl.read_csv("../../input/test.csv").drop("id")
orig = pl.read_parquet("../../input/original.parquet")
orig = orig.with_columns(
    pl.when(pl.col("y") == "yes").then(1)
      .when(pl.col("y") == "no").then(0)
      .otherwise(None)
      .alias("y")
)

y_tr = train["y"].cast(pl.Int8)
y_orig = orig["y"].cast(pl.Int8)
y_merged = pl.concat([y_tr, y_orig], how="vertical")

train = train.drop("y")
orig = orig.drop("y")

CATS = [col for col in train.columns if train[col].dtype == pl.Utf8]
NUMS = [col for col in train.columns if train[col].dtype != pl.Utf8]

In [5]:
# === 全データを結合 ===
all_data = pl.concat([train, test, orig], how="vertical")
cat_exprs = [
    pl.col(c)
    .cast(pl.Categorical)
    .to_physical()
    .rank("dense")
    .cast(pl.Int32)
    .alias(c)
    for c in CATS
]
num_df = all_data.select(NUMS)
cat_df = all_data.select(
    [pl.col(c).cast(pl.Utf8).cast(pl.Categorical) for c in CATS]
)

In [6]:
# === NUM → CAT ===
SIZES = {}

num2cat_exprs = [
    pl.col(c)
    .cast(pl.Utf8)
    .cast(pl.Categorical)
    .to_physical()
    .cast(pl.Int32).alias(f"{c}2")
    for c in NUMS
]

num_df2 = all_data.select(num2cat_exprs)
NUMS2 = num_df2.columns

all_data = all_data.with_columns(cat_exprs + num2cat_exprs)

SIZES = all_data.select(
    [pl.col(col)
     .n_unique()
     .alias(col) for col in CATS + NUMS2]
).to_dicts()[0]

print(SIZES)

{'job': 12, 'marital': 3, 'education': 4, 'default': 2, 'housing': 2, 'loan': 2, 'contact': 3, 'month': 12, 'poutcome': 4, 'age2': 78, 'balance2': 8590, 'day2': 31, 'duration2': 1824, 'campaign2': 52, 'pdays2': 628, 'previous2': 54}


In [7]:
# === 2Comboのペアを作成 ===
pairs = list(combinations(CATS + NUMS2, 2))

combo_exprs = [(pl.col(c1) * SIZES[c2] + pl.col(c2))
               .alias(f"{c1}_{c2}") for c1, c2 in pairs]

COMBO = [f"{c1}_{c2}" for c1, c2 in pairs]

combo2_df = all_data.with_columns(combo_exprs)

print(f"Created {len(combo_exprs)} new columns")

Created 120 new columns


In [8]:
# === Targetをoriginalのものにする ===
tr_df = combo2_df[:len(train)]
test_df = combo2_df[len(train):len(train)+len(test)]

orig_df = combo2_df[len(train)+len(test):]
orig_df = orig_df.with_columns(y_orig.alias("target"))

te_cols = CATS + NUMS2 + COMBO

all_cols = [f"target_mean_by_{c}2" for c in te_cols]

N_tr, N_te = tr_df.height, test_df.height

te_train = {c: np.zeros(N_tr, dtype=np.float32) for c in all_cols}
te_test = {c: np.zeros(N_te, dtype=np.float32) for c in all_cols}

for col in tqdm(te_cols):
    base = orig_df.select([
        pl.col("target").mean().alias("mean")
    ]).to_dicts()[0]

    fill_map = {}
    name = f"target_mean_by_{col}2"
    fill_map[name] = float(base["mean"])

    grouped_df = (
        orig_df.select([col, "target"])
        .group_by(col)
        .agg(pl.col("target").mean().alias(f"target_mean_by_{col}2"))
    )

    val_mat = (
        tr_df.join(
            grouped_df.select([name, col]),
            on=col,
            how="left"
        )
        .select(name)
        .with_columns(
            [
                pl.col(name).fill_null(fill_map[name]).alias(name)
            ]
        )
        .to_numpy()
        .astype(dtype=np.float32, copy=False)
        .ravel()
    )

    te_train[name] = val_mat

    # 4. テストデータも同様にjoin
    test_mat = (
        test_df.join(
            grouped_df.select([name, col]),
            on=col,
            how="left"
        )
        .select(name)
        .with_columns(
            [
                pl.col(name).fill_null(fill_map[name]).alias(name)
            ]
        )
        .to_numpy()
        .astype(dtype=np.float32, copy=False)
        .ravel()
    )

    te_test[name] += test_mat

te_tr = pl.DataFrame(te_train).with_columns([
    pl.col(col) for col in te_train.keys()
])
te_test = pl.DataFrame(te_test).with_columns([
    pl.col(col) for col in te_test.keys()
])
te_orig = pl.concat([te_tr, te_test], how="vertical")

print(f"Created {len(te_orig.columns)} new columns")

  0%|          | 0/136 [00:00<?, ?it/s]

Created 136 new columns


In [9]:
# === Target Encoding ===
tr_df = tr_df.with_columns(y_tr.alias("target"))

te_df = target_encoding(tr_df, test_df, key_cols=te_cols, stats=("mean", ))

print(f"Created {len(te_df.columns)} new columns")

0it [00:00, ?it/s]

  0%|          | 0/136 [00:00<?, ?it/s]

  0%|          | 0/136 [00:00<?, ?it/s]

  0%|          | 0/136 [00:00<?, ?it/s]

  0%|          | 0/136 [00:00<?, ?it/s]

  0%|          | 0/136 [00:00<?, ?it/s]

Created 136 new columns


In [10]:
# === Count Encoding
all_data = combo2_df[:len(train) + len(test)]
ce_cols = te_cols
ce_dict = {f"{col}_ce": np.zeros(all_data.height) for col in ce_cols}

for col in tqdm(ce_cols):
    counts = combo2_df.group_by(col).agg(pl.len().alias(f"{col}_ce"))
    joined_df = combo2_df.join(counts, on=col, how="left")
    ce_dict[f"{col}_ce"] = joined_df[f"{col}_ce"]

ce_df = pl.DataFrame(ce_dict).with_columns([
        pl.col(col).cast(pl.Float32) for col in ce_dict.keys()
])

print(f"Created {len(ce_df.columns)} new columns")

  0%|          | 0/136 [00:00<?, ?it/s]

Created 136 new columns


In [11]:
# === Dataの統合 ===
all_data = pl.concat([
    num_df,
    te_df,
    te_orig,
    ce_df
], how="horizontal")

In [12]:
# === row_id を追加 ===
all_data = all_data.with_row_index("row_id")

# === Downcast ===
all_data = downcast(all_data)

# === データを分割 ===
tr_df = all_data[:len(train)]
test_df = all_data[len(train):len(train)+len(test)]

# === targetを追加 ===
tr_df = tr_df.with_columns(y_tr.alias("target"))

In [13]:
# Add Fold Col
folds_path = "../../artifacts/folds/folds.parquet"
pairs = [
    ("skf/k=5/s=42@train", "5fold-s42")
]
cfgs = [c for c, _ in pairs]
rename_map = {c: n for c, n in pairs}

# folds をまとめて読み → ワイド化（cfg列を列見出しに）→ 列名をfold_nameにリネーム
folds_wide = (
    pl.scan_parquet(folds_path)
      .filter(pl.col("cfg").is_in(cfgs))
      .unique(subset=["row_id", "cfg"], keep="last")
      .select(["row_id", "cfg", "fold"])
      .collect(engine="streaming")
      .pivot(values="fold", index="row_id", on="cfg", aggregate_function="first")
      .rename(rename_map)
      .with_columns(pl.col("row_id").cast(pl.Int32))
      .with_columns([pl.all().exclude("row_id").cast(pl.Int8)])  # 型を軽く
)

# tr_df が DataFrame の場合
tr_df = tr_df.join(folds_wide, on="row_id", how="left")

In [14]:
# === 特徴量エンジニアリング後の情報 ===
train_mem, test_mem, n_cat = check_info(tr_df, test_df)

=== Shape & Memory ===
Train Shape: (750000, 418), Test Shape: (250000, 416)
Train Memory: 1.16 GB, Test Memory: 0.39 GB

=== DTypes ===
UInt32: 1
Int32: 7
Float32: 408
Int8: 2


In [16]:
# === Save Overall Data ===
tr_path = FEATURE_DIR / "train.parquet"
test_path = FEATURE_DIR / "test.parquet"

tr_df.write_parquet(tr_path)
test_df.write_parquet(test_path)

print(f"tr_df saved successfully to {tr_path}")
print(f"test_df saved successfully to {test_path}")

tr_df saved successfully to ../../artifacts/features/base/044/train.parquet
test_df saved successfully to ../../artifacts/features/base/044/test.parquet


## Meta dataを保存

In [27]:
tr_path = FEATURE_DIR / "train.parquet"
test_path = FEATURE_DIR / "test.parquet"

In [28]:
JST = timezone(timedelta(hours=9))
meta = {
    "data_id": ID,
    "created_at": datetime.now(JST).isoformat(),
    "train_paths": [str(tr_path)],
    "test_paths": [str(test_path)],
    "level": LEVEL,
    "train_shape": [tr_df.height, tr_df.width],
    "test_shape": [test_df.height, test_df.width],
    "memory": {
        "train": train_mem,
        "test": test_mem
    },
    "fold_column": pairs,
    "cat_cols": n_cat if n_cat else None
}

with open(f"{FEATURE_DIR}/meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)